# Chapter 13 — Error Correcting Codes

Computer exercises from Section 13.13: linear block code encoding and syndrome decoding, Hamming codes in AWGN, and LDPC codes.

```{admonition} Running these exercises
:class: tip
Every figure on this page is produced by the code directly above it, and the
code runs when the site is built. Use the **launch button** (the rocket icon) at the
top of the page to open this notebook in Google Colab or a live Binder session,
or hit **live code** to run and edit the cells right here in the browser.
```


## 13.13.1 Computer Exercise 13.1: Block Decoding
In the first experiment, we provide a program to decode the (6, 3) linear block code of Example 13.1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(447)  # fixed seed so the figures below are reproducible


In [ ]:
# Python program to illustrate encoding and decoding of (6,3) block code in Example 13.1

# Code Generator
G = np.array([[1, 0, 0, 1, 0, 1], [0, 1, 0, 0, 1, 1], [0, 0, 1, 1, 1, 0]])

# Parity Check Matrix
H = np.array([[1, 0, 1], [0, 1, 1], [1, 1, 0], [1, 0, 0], [0, 1, 0], [0, 0, 1]]).T

# List of correctable errors
E = np.array([[0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0],
[0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 1], [1, 0, 0, 0, 1, 0]])

K = E.shape[0]
Syndrome = np.mod(np.dot(E, np.transpose(H)), 2) # Find Syndrome List
r = np.array([1, 1, 1, 0, 1, 1]) # Received codeword
print('Syndrome Error Pattern')
print(np.concatenate((Syndrome, E), axis=1)) # Display Syndrome List and Error Patterns
x = np.mod(np.dot(r, np.transpose(H)), 2) # Compute Syndrome

idxe = None
for kk in range(K):
    if np.array_equal(Syndrome[kk, :], x):
        idxe = kk # Find the Syndrome Index

syndrome = Syndrome[idxe, :] # Display the Syndrome
error = E[idxe, :]
cword = np.bitwise_xor(r, error) # Error Correction

print('Syndrome:', syndrome)
print('Error:', error)
print('Corrected Codeword:', cword)

The execution of this Python program will generate the following results, which include the erroneous codeword, the syndrome, the error pattern, and the corrected codeword.

``` Python
Syndrome Error Pattern
[[0 0 0 0 0 0 0 0 0]
 [1 0 1 1 0 0 0 0 0]
 [0 1 1 0 1 0 0 0 0]
 [1 1 0 0 0 1 0 0 0]
 [1 0 0 0 0 0 1 0 0]
 [0 1 0 0 0 0 0 1 0]
 [0 0 1 0 0 0 0 0 1]
 [1 1 1 1 0 0 0 1 0]]
Syndrome: [0 1 1]
Error: [0 1 0 0 0 0]
Corrected Codeword: [1 0 1 0 1 1]
```

In our next exercise, we provide a program to decode the (7, 4) Hamming code of Example 13.3.

In [ ]:
# Python Program to illustrate encoding and decoding of Hamming (7,4) code

# Code Generating Matrix
G = np.array([[1, 0, 0, 0, 1, 0, 1],
[0, 1, 0, 0, 1, 1, 1],
[0, 0, 1, 0, 1, 1, 0],
[0, 0, 0, 1, 0, 1, 1]])

# Parity Check Matrix
H = np.concatenate((G[:, 4:7].T, np.eye(3, dtype=int)), axis=1)

#  List of correctable errors
E = np.array([[1, 0, 0, 0, 0, 0, 0],
[0, 1, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0],
[0, 0, 0, 1, 0, 0, 0],
[0, 0, 0, 0, 1, 0, 0],
[0, 0, 0, 0, 0, 1, 0],
[0, 0, 0, 0, 0, 0, 1]])

K = E.shape[0]

# Find Syndrome List
Syndrome = np.mod(np.dot(E, H.T), 2)

# Received codeword
r = np.array([1, 0, 1, 0, 1, 1, 1])

print('Syndrome', 'Error Pattern')
print(np.hstack((Syndrome, E)))

# Compute syndrome
x = np.mod(np.dot(r, H.T), 2)

# Find the syndrome index
for kk in range(K):
    if np.array_equal(Syndrome[kk], x):
        idxe = kk

syndrome = Syndrome[idxe]
print('Syndrome:', syndrome)

error = E[idxe]
print('Error Pattern:', error)

# Error correction
cword = np.bitwise_xor(r, error)
print('Corrected Codeword:', cword)

Executing Python program will generate for an erroneous codeword r its syndrome, the error pattern, and the corrected codeword:

``` Python
Syndrome Error Pattern
[[1 0 1 1 0 0 0 0 0 0]
 [1 1 1 0 1 0 0 0 0 0]
 [1 1 0 0 0 1 0 0 0 0]
 [0 1 1 0 0 0 1 0 0 0]
 [1 0 0 0 0 0 0 1 0 0]
 [0 1 0 0 0 0 0 0 1 0]
 [0 0 1 0 0 0 0 0 0 1]]
Syndrome: [1 0 0]
Error Pattern: [0 0 0 0 1 0 0]
Corrected Codeword: [1 0 1 0 0 1 1]
```

## 13.13.2 Computer Exercise 13.2: Error Correction in AWGN Channels
<span style="color:red"> <b>*Note: text on p.952, 954 skipped*</b></span>

In [ ]:
'''Simulation of the Hamming (7,4) code performance
under polar signaling in AWGN channel and performance
comparison with uncoded polar signaling'''
# Code Generator
G = np.array([
    [1, 0, 0, 0, 1, 0, 1],
    [0, 1, 0, 0, 1, 1, 1],
    [0, 0, 1, 0, 1, 1, 0],
    [0, 0, 0, 1, 0, 1, 1]])
# Parity Check Matrix
H = np.array([
    [1, 1, 1, 0, 1, 0, 0],
    [0, 1, 1, 1, 0, 1, 0],
    [1, 1, 0, 1, 0, 0, 1]])
# Error patterns
E = np.array([
    [1, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0, 0],
    [0, 0, 0, 1, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 1],
    [0, 0, 0, 0, 0, 0, 0]])
K2 = E.shape[0]
Syndrome = np.mod(np.matmul(E, H.T), 2)  # Syndrome list
L1 = 500000
K = 4 * L1  # Decide how many codewords
sig_b = np.round(np.random.rand(K))
sig_2 = np.reshape(sig_b, (4, L1), order='F')  # 4 per column for FEC
xig_1 = np.mod(np.matmul(G.T, sig_2), 2)  # Encode column by column
xig_2 = 2 * np.reshape(xig_1, (7 * L1,), order='F') - 1  # P/S conversion
AWnoise1 = np.random.randn(7 * L1)  # Generate AWGN for coded Tx
AWnoise2 = np.random.randn(4 * L1)  # Generate AWGN for uncoded Tx
# Change SNR and compute BER's
num_ex = 14 # number of test points
BER_coded = np.zeros(num_ex)
BER_uncode = np.zeros(num_ex)
for ii in range(num_ex):
    SNRdb = ii + 1
    SNR = 10 ** (SNRdb * 0.1)
    xig_n = np.sqrt(SNR * 4 / 7) * xig_2 + AWnoise1  # Add AWGN and adjust SNR
    rig_1 = (1 + np.sign(xig_n)) / 2  # Hard decisions
    r = np.reshape(rig_1, (7, L1), order='F').T  # S/P to form 7 bit codewords
    x = np.mod(np.matmul(r, H.T), 2)  # generate error syndromes
    sigcw = np.zeros((4, L1))
    for k1 in range(L1):
        idxe = np.flatnonzero((Syndrome == x[k1]).all(1))[0]  # find the Syndrome index
        error = E[idxe]  # look up the error pattern
        cword = np.logical_xor(r[k1], error)  # error correction
        sigcw[:, k1] = cword[:4]  # keep the message bits
    cw = np.reshape(sigcw, K, order='F')
    BER_coded[ii] = np.sum(np.abs(cw - sig_b)) / K  # Coded BER on info bits
    xig_3 = 2 * sig_b - 1  # Polar signaling
    xig_m = np.sqrt(SNR) * xig_3 + AWnoise2  # Add AWGN and adjust SNR
    rig_1 = (1 + np.sign(xig_m)) // 2  # Hard decision
    BER_uncode[ii] = np.sum(np.abs(rig_1 - sig_b)) / K  # Compute BER
EboverN = np.arange(num_ex) - 3 # Need to note that SNR = Eb/N + 3
print(EboverN)

In [ ]:
# replace 0 values by nan so that they can be masked in plots
BER_coded[BER_coded==0] = np.nan
BER_uncode[BER_uncode==0] = np.nan
plt.semilogy(EboverN,BER_uncode,'b-',label='Uncoded',linewidth=2)
plt.semilogy(EboverN,BER_coded,'b--',label='Coded',linewidth=2)
plt.legend(loc='upper right')
plt.xlabel(r'$E_b/N$, dB')
plt.ylabel('Bit Error Rate (BER)')
plt.grid();plt.show()

<center><b>Figure 13.24</b> Comparison of BERs of uncoded polar signaling transmission and polar signaling transmission of Hamming (7, 4) encoded (dashed) and uncoded (solid) message bits. </center>

In [ ]:
print(BER_coded)

## 13.13.3 Computer Exercise 13.3(New): Error Correction in AWGN Channels with General Hamming Code

In [ ]:
def generator_polynomial_hamming(m):
    # This program returns the generator polynomials for hamming
    # code with given number of parity bits m
    gene_poly = {
        3: np.array([1, 0, 1, 1]),
        4: np.array([1, 0, 0, 1, 1]),
        5: np.array([1, 0, 0, 1, 0, 1]),
        6: np.array([1, 0, 0, 0, 0, 1, 1]),
        7: np.array([1, 0, 0, 0, 1, 0, 0, 1]),
        8: np.array([1, 0, 0, 0, 1, 1, 1, 0, 1]),
        9: np.array([1, 0, 0, 0, 0, 1, 0, 0, 0, 1]),
        10: np.array([1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1]),
        11: np.array([1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1]),
        12: np.array([1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1]),
        13: np.array([1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1]),
        14: np.array([1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1]),
        15: np.array([1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1]),
        16: np.array([1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1])
    }

    return gene_poly.get(m)

def hamming_general_encoder(m, data):
    # This program use (2**m-1, 2**m-1-m, m) hamming code for
    # encode, m is the number of parity bits

    # Determine the code length and number of message bits
    n = 2**m - 1 # Code length
    k = 2**m - m - 1  # Number of message bits

    # Find generator polynomial for hamming code
    gp = generator_polynomial_hamming(m)
    gp_pad = np.zeros(n, dtype=int)
    gp_pad[:len(gp)] = gp # pad zeros at the end of generator polynomial

    # Create non-systematic generating matrix by circular shift of gp_pad
    G_nonSystem = np.array([np.roll(gp_pad, i) for i in range(k)], dtype=int)

    # Create systematic generating matrix G by XOR on selected rows in
    # G_nonSystem so that the left part of G is an identity matrix
    G = np.zeros([k,n], dtype=int)
    for row1 in range(k):
      G[row1,:] = G_nonSystem[row1,:]
      for row2 in range(row1+1,k):
        if G[row1,row2] == 1:
          G[row1,:] = np.logical_xor(G[row1,:], G_nonSystem[row2,:])
    P = G[:,k:]

    # Create Parity Check Matrix H
    H = np.concatenate((P.T, np.eye(m, dtype=int)), axis=1)

    # Encode data using G
    data_mat = np.reshape(data, (k,-1), order='F') # k bits per column
    encoded_mat = np.matmul(G.T, data_mat) % 2
    encoded_data = np.reshape(encoded_mat, (1,-1), order='F')

    return encoded_data, G, H

def hamming_general_decoder(m, H, rdata):
    # This program decode rdata with (2**m-1, 2**m-1-m, m) hamming
    # code, m is the number of parity bits

    # Determine the code length and number of message bits
    n = 2**m - 1 # Code length
    k = 2**m - m - 1  # Number of message bits

    # Error patterns
    E = np.vstack((np.eye(n,dtype=int), np.zeros((1,n),dtype=int)))
    Syndrome = np.matmul(E, H.T) % 2  # Syndrome list

    # Reshape rdata to form n bit codewords
    L1 = len(rdata) // n
    rdata_mat = np.reshape(rdata, (n,L1), order='F').T
    Error_syndrome = np.matmul(rdata_mat, H.T) % 2 # error syndromes
    cdata_mat = np.zeros((k, L1), dtype=int)
    for k1 in range(L1):
      # find the Syndrome index
      idxe = np.flatnonzero((Syndrome == Error_syndrome[k1]).all(1))[0]
      error = E[idxe] # look up the error pattern
      cword = np.logical_xor(rdata_mat[k1], error)  # error correction
      cdata_mat[:, k1] = cword[:k]  # keep the message bits
    cdata = np.reshape(cdata_mat, (1,-1), order='F')

    return cdata

In [ ]:
'''Simulation of the Hamming (15,11) code performance
under polar signaling in AWGN channel and performance
comparison with uncoded polar signaling'''
m = 4 # number of parity bits; feasible values are m={3,4,...,16}
n = 2**m - 1 # Code length
k = 2**m - m - 1  # Number of message bits

L1 = 500000;  K = k * L1
sig_b = np.round(np.random.rand(K))
sig_2 = np.reshape(sig_b, (k, L1), order='F')  # k per column for FEC
xig_1, G, H = hamming_general_encoder(m, sig_b) # Encode column by column
xig_2 = 2 * np.reshape(xig_1, (n * L1,), order='F') - 1  # P/S conversion
AWnoise1 = np.random.randn(n * L1)  # Generate AWGN for coded Tx
AWnoise2 = np.random.randn(k * L1)  # Generate AWGN for uncoded Tx

# Change SNR and compute BER's
num_ex = 13 # number of test points
BER_coded = np.zeros(num_ex)
BER_uncode = np.zeros(num_ex)
for ii in range(num_ex):
    SNRdb = ii + 1
    SNR = 10 ** (SNRdb * 0.1)
    xig_n = np.sqrt(SNR * k / n) * xig_2 + AWnoise1  # Add AWGN and adjust SNR
    rig_1 = (1 + np.sign(xig_n)) / 2  # Hard decisions
    cw = hamming_general_decoder(m, H, rig_1)
    BER_coded[ii] = np.sum(np.abs(cw - sig_b)) / K  # Coded BER on info bits
    xig_3 = 2 * sig_b - 1  # Polar signaling
    xig_m = np.sqrt(SNR) * xig_3 + AWnoise2  # Add AWGN and adjust SNR
    rig_1 = (1 + np.sign(xig_m)) // 2  # Hard decision
    BER_uncode[ii] = np.sum(np.abs(rig_1 - sig_b)) / K  # Compute BER
EboverN = np.arange(num_ex) - 3 # Need to note that SNR = Eb/N + 3

In [ ]:
# replace 0 values by nan so that they can be masked in plots
BER_coded[BER_coded==0] = np.nan
BER_uncode[BER_uncode==0] = np.nan
plt.semilogy(EboverN,BER_uncode,'b-',label='Uncoded',linewidth=2)
plt.semilogy(EboverN,BER_coded,'b--',label='Coded',linewidth=2)
plt.legend(loc='upper right')
plt.xlabel(r'$E_b/N$, dB')
plt.ylabel('Bit Error Rate (BER)')
plt.grid();plt.show()

<center>Comparison of BERs of uncoded polar signaling transmission and polar signaling transmission of Hamming (15, 11) encoded (dashed) and uncoded (solid) message bits.</center>

## 13.13.4 Computer Exercise 13.4(New): Error Correction in AWGN Channels with LDPC Code

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# pyldpc is listed in the course requirements.txt, so it is already installed
# when this page is built and in a Binder session. Google Colab does not ship
# it, so install it there on the fly.
try:
    from pyldpc import make_ldpc, encode, decode, get_message
except ModuleNotFoundError:
    %pip install -q pyldpc
    from pyldpc import make_ldpc, encode, decode, get_message

In [ ]:
'''Simulation of the LPDC code performance
under BPSK polar signaling in AWGN channel and performance
comparison with uncoded polar signaling'''

n = 260 # code length divisible by 4
d_v = 2 # Number of 1s per column in the H matrix
d_c = 4 # Number of 1s per row in the H matrix

# Create LDPC parity-check matrix H and coding matrix G
H, G = make_ldpc(n, d_v, d_c, systematic=True, sparse=True)
n, k = G.shape

L1 = 250   # number of code blocks. Raise it (500-5000) for smoother curves;
           # the decoder is the slow part, and this page is re-executed on
           # every site build.
K = k * L1
sig_b = np.round(np.random.rand(K))
sig_2 = np.reshape(sig_b, (k, L1), order='F')  # k per column for FEC
xig_1 = np.matmul(G, sig_2) % 2 # Encode column by column
xig_2 = -2 * np.reshape(xig_1, (n * L1,), order='F') + 1  # P/S conversion
AWnoise1 = np.random.randn(n * L1)  # Generate AWGN for coded Tx
AWnoise2 = np.random.randn(k * L1)  # Generate AWGN for uncoded Tx

# Change SNR and compute BER's
num_ex = 16 # number of test points
BER_coded = np.zeros(num_ex)
BER_uncode = np.zeros(num_ex)
for ii in range(num_ex):
    SNRdb = ii + 1
    SNR = 10 ** (SNRdb * 0.1)
    xig_n = np.sqrt(SNR * k / n) * xig_2 + AWnoise1  # Add AWGN and adjust SNR
    xig_mat = np.reshape(xig_n, (n, L1), order='F')
    rig_l = decode(H, xig_mat, 0)  # Decisions
    cw_mat = np.zeros((k,L1),dtype=int)
    for i in range(L1):
      cw_mat[:,i] = get_message(G, rig_l[:,i])
    cw = np.reshape(cw_mat, K, order='F')
    BER_coded[ii] = np.sum(np.abs(cw - sig_b)) / K  # Coded BER on info bits

    xig_3 = 2 * sig_b - 1  # Polar signaling
    xig_m = np.sqrt(SNR) * xig_3 + AWnoise2  # Add AWGN and adjust SNR
    rig_1 = (1 + np.sign(xig_m)) // 2  # Hard decision
    BER_uncode[ii] = np.sum(np.abs(rig_1 - sig_b)) / K  # Compute BER
EboverN = np.arange(num_ex) - 3 # Need to note that SNR = Eb/N + 3

In [ ]:
# replace 0 values by nan so that they can be masked in plots
BER_coded[BER_coded==0] = np.nan
BER_uncode[BER_uncode==0] = np.nan
plt.semilogy(EboverN,BER_uncode,'k-',label='Uncoded')
plt.semilogy(EboverN,BER_coded,'k--',label='Coded')
plt.legend(loc='upper right')
plt.xlabel(r'$E_b/N$, dB')
plt.ylabel('Bit Error Rate (BER)')
plt.grid();plt.show()